### 0: Downloading Programmes

In [11]:
import gurobipy as gp
from gurobipy import GRB
import pyomo.environ as pyo
from pyomo.opt import SolverFactory
import pandas as pd
from pathlib import Path
from collections import defaultdict


### 1: Dataframes

In [62]:
base = Path("output_folder")


df_resources = pd.read_csv(base / "resources.csv") # resource ID and resource type
df_constraints = pd.read_csv(base / "constraints.csv") # constraint ID and type, Name, required?, weight and cost function

df_times = pd.read_csv(base / "times.csv").sort_values("order_index") # timeslots and their timegroups (days)



# df_events = event with its duration, course, groups, role, classes and teacher)

events = pd.read_csv(base / "events.csv") 
event_groups = pd.read_csv(base / "event_eventgroup_membership.csv")
event_resources = pd.read_csv(base / "event_resources.csv")

# Events each group is part of
groups_per_event = (
    event_groups
    .groupby("event_id")["event_group_id"]
    .apply(list)
    .reset_index(name="groups")
)

# Teacher of each event
teachers_per_event = event_resources[event_resources["role"] == "Teacher"]
teachers_per_event = ( teachers_per_event[["event_id", "reference_resource_id"]]
    .groupby("event_id")["reference_resource_id"]
    .apply(list)
    .reset_index(name="teachers")
) 

#Classes in each event
classes_per_event = event_resources[event_resources["role"].astype(str).str.startswith("Class")]
classes_per_event = ( classes_per_event[["event_id", "reference_resource_id"]]
    .groupby("event_id")["reference_resource_id"]
    .apply(list)
    .reset_index(name="classes")
)

# Role of each event
role_per_event = event_resources[event_resources["resource_type_ref"] == "Room"]
role_per_event = (role_per_event[["event_id", "role"]]
    .groupby("event_id")["role"]
    .first()
    .reset_index(name="role")
)

df_events = (
    events.rename(columns={"course_ref": "course_reference"})[["event_id", "duration", "course_reference"]]
    .merge(groups_per_event, on="event_id", how="left")
    .merge(role_per_event, on="event_id", how="left")
    .merge(classes_per_event, on="event_id", how="left")
    .merge(teachers_per_event, on="event_id", how="left")
) # event with its duration, course, groups, role, classes and teacher)



resource_groups = pd.read_csv(base / "resource_group_membership.csv")

# Keep only what we need from memberships and aggregate group memberships
groups_per_resource = (
    resource_groups
    .groupby("resource_id")["resource_group_id"]
    .apply(list)
    .reset_index(name="resource_groups")
)

# Merge onto resources
df_res_with_groups = (
    df_resources[["resource_id", "resource_type_ref"]]
    .merge(groups_per_resource, on="resource_id", how="left")
)

# Replace NaN with empty list (resources that have no group membership)
df_res_with_groups["resource_groups"] = df_res_with_groups["resource_groups"].apply(
    lambda x: x if isinstance(x, list) else []
)

# Split into separate tables
df_teachers = df_res_with_groups[df_res_with_groups["resource_type_ref"] == "Teacher"].reset_index(drop=True) # teacher id, resource type and list of groups
df_classes  = df_res_with_groups[df_res_with_groups["resource_type_ref"] == "Class"].reset_index(drop=True) # class id, resource type and list of groups
df_rooms    = df_res_with_groups[df_res_with_groups["resource_type_ref"] == "Room"].reset_index(drop=True) # room id, resource type and list of groups


df_res_long = (
    df_resources[["resource_id", "resource_type_ref"]]
    .merge(resource_groups[["resource_id", "resource_group_id"]], on="resource_id", how="left")
    .rename(columns={"resource_group_id": "resource_group"})
)
df_teachers_long = df_res_long[df_res_long["resource_type_ref"] == "Teacher"] # teacher id, resource type and seperate groups
df_classes_long  = df_res_long[df_res_long["resource_type_ref"] == "Class"] # class id, resource type and seperate groups
df_rooms_long    = df_res_long[df_res_long["resource_type_ref"] == "Room"] # room id, resource type and seperate groups




df_constraint_applied = pd.read_csv(base / "constraint_applies_to.csv")
df_constraint_params = pd.read_csv(base / "constraint_params.csv")

df_constraint_params = df_constraint_params.merge(
    df_constraints[["constraint_id", "constraint_type"]],
    on="constraint_id",
    how="left"
)

df_constraint_params = df_constraint_params.merge(
    df_constraint_applied[["constraint_id", "reference"]],
    on="constraint_id",
    how="left"
)

In [63]:
#print(df_resources)
#print(df_constraints)

#print(df_times)
#print(df_events)
#print(df_rooms)
#print(df_teachers)
#print(df_classes)

#df_constraint_params



### 2a: Model setup and helpers

In [95]:

def as_list(x):
    if x is None:
        return []
    if isinstance(x, (list, tuple, set)):
        return list(x)
    return [x]




### Sets

E = df_events["event_id"].tolist() # All events
T = df_times["time_id"].tolist() # All timeslots
R = df_rooms["resource_id"].tolist() # All rooms

all_teachers = df_teachers["resource_id"].tolist()
all_classes = df_classes["resource_id"].tolist()


# event -> duration/teachers/classes (lists)
event_to_duration = dict(zip(df_events["event_id"], df_events["duration"].astype(int))) # duration of each event
event_to_teachers = dict(zip(df_events["event_id"], df_events["teachers"])) # teachers for each event
event_to_classes  = dict(zip(df_events["event_id"], df_events["classes"])) # classes for each event


# event -> room group of each event
event_to_roomgroup = dict(zip(df_events["event_id"], df_events["role"])) 

# room -> room group of each room
room_groups = {"gr_Rooms_X", "gr_Rooms_Y", "gr_Rooms_Z"}
room_to_group = (
    df_rooms_long[df_rooms_long["resource_group"].isin(room_groups)]
    .set_index("resource_id")["resource_group"]
    .to_dict()
)

# room group -> rooms in that group
group_to_rooms = defaultdict(list)
for room, grp in room_to_group.items():   # room_to_group: room -> gr_Rooms_X/Y/Z
    group_to_rooms[grp].append(room)


T_index = {t:i for i,t in enumerate(T)}

df_unavail_times = df_constraint_params[
    (df_constraint_params["constraint_type"] == "AvoidUnavailableTimesConstraint") &
    (df_constraint_params["path"] == "Times/Time")
].copy()

df_unavail_times = df_unavail_times[["reference", "value", "constraint_id"]]

# df_unavail_times has: reference (teacher), value (times teacher not available), constraint_id
unavailability = (
    df_unavail_times
    .groupby("constraint_id")
    .agg(
        teacher=("reference", "first"),
        times=("value", lambda s: sorted(set(s), key=lambda t: T_index[t]))
    )
    .to_dict(orient="index")
)

In [ ]:
### Feasible Start times for events:

#T_index = {t:i for i,t in enumerate(T)}

time_to_day = dict(zip(df_times["time_id"], df_times["day_ref"]))

# Build feasible starts per event (no crossing day boundary) (implicitely covers constraints 5,9,10,11,12)
feasible_starts = {}
for e in E:
    d = int(event_to_duration[e]) # number of periods needed
    starts = []
    for ts in T: # for all possible start times
        i0 = T_index[ts]
        # must have enough periods remaining in the global list
        if i0 + d - 1 >= len(T):
            continue
        day0 = time_to_day[ts]
        block = T[i0:i0 + d]
        # enforce all times in the block are same day
        if all(time_to_day[t] == day0 for t in block):
            starts.append(ts)
    feasible_starts[e] = starts

# list of occupied times for each (event, starttime) pair
occ_times = {}
for e in E:
    d = int(event_to_duration[e])
    for ts in feasible_starts[e]:
        i0 = T_index[ts]
        occ_times[(e, ts)] = T[i0:i0 + d]

In [97]:
# Copy so you can compare before/after if you want
feasible_starts_filtered = {}

for e in E:
    starts_ok = []
    for ts in feasible_starts[e]:
        occ = occ_times[(e, ts)]  # occupied periods if e starts at ts

        # Check teacher-specific unavailability constraints
        violates = False
        for info in unavailability.values():
            teacher = info["teacher"]
            unavail_set = set(info["times"])
            if teacher in as_list(event_to_teachers[e]):
                # if any occupied time is unavailable -> illegal
                if any(t in unavail_set for t in occ):
                    violates = True
                    break

        if not violates:
            starts_ok.append(ts)

    feasible_starts_filtered[e] = starts_ok


# Replace
feasible_starts = feasible_starts_filtered
#feasible_starts


# rebuild occ_times
occ_times = {}
for e in E:
    d = int(event_to_duration[e])
    for ts in feasible_starts[e]:
        i0 = T_index[ts]
        occ_times[(e, ts)] = T[i0:i0 + d]

# rebuild occupies_at_time
occupies_at_time = {t: [] for t in T}
for e in E:
    for ts in feasible_starts[e]:
        for t in occ_times[(e, ts)]:
            occupies_at_time[t].append((e, ts))

In [98]:
### Feasible Room Groups
role_to_roomgroup = {
    "gr_Room_X": "gr_Rooms_X",
    "gr_Room_Y": "gr_Rooms_Y",
    "gr_Room_Z": "gr_Rooms_Z",
}

feasible_rooms = {}
for e in E:
    role = event_to_roomgroup[e]                  # e -> gr_Room_X/Y/Z
    rg = role_to_roomgroup[role]             # -> gr_Rooms_X/Y/Z
    feasible_rooms[e] = group_to_rooms[rg]   # list of rooms in that group

In [74]:
#### Hard Coding Feasible Starts and feasible rooms

time_to_day = dict(zip(df_times["time_id"], df_times["day_ref"]))

all_starts = {e: list(T) for e in E}

# Build occupied-time blocks for every candidate start
occ_times = {}
for e in E:
    d = int(event_to_duration[e])
    for ts in T:
        i0 = T_index[ts]
        if i0 + d - 1 < len(T):
            occ_times[(e, ts)] = T[i0:i0 + d]
        else:
            occ_times[(e, ts)] = []


occupies_at_time = {t: [] for t in T}
for e in E:
    for ts in T:
        for t in occ_times[(e, ts)]:
            occupies_at_time[t].append((e, ts))

feasible_starts = all_starts

feasible_rooms = {e: list(R) for e in E}

### 2b: Pyomo Model part 1: Sets and Decision Variables

Constraints:

Part 1: Event -> time (hard constraints):

4) AssignTimes: [gr_EventsGeneral] Every event must be assigned a time
5) Do Not Split Events: [gr_EventsGeneral] Events must inhabit consecutive periods
9) PreferredTimes: [gr_EventsDuration1] Events with duration 1 must only be allocated to times in groups gr_TimesDurationTwo
10) PreferredTimesDurationTwo: [gr_EventsDuration1] ...
11) PreferredTimesDurationThree: [gr_EventsDuration1] ...
12) PreferredTimesDurationFour: [gr_EventsDuration1] ...
13) EventSpreadingConstraint: [gr_001 - gr_150] events from each goup have a maximum and minimum specified number of events in each time group [monday, tuesday etcc...] (here max 1 per day) (event group given in course ID)
14) NoResourceClashes: No clashes for any of the resources
15) Unavailabilities_T07: [T07] selected Teacher cannot be allocated event at any of the given times, 
...
25) Unavailabilities_T17: ...

Extra: N(R_X) <= 1,  N(R_y) <= 2,  N(R_X) <= 9 in each timeslot to ensure feasability of room assignments


Part 2: Event -> time (add soft constraints):

26) JumpPeriods_Classes: [gr_classes] classes should have a minimum of 0 and maximum of 0 idle timeperiods (no timeperiod where there is no event but there is an earlier and later event on the same day) on every given time group (day) (soft) 
27) JumpPeriods_Teachers: [gr_teachers] teachers should have a minimum of 0 and maximum of 0 idle timeperiods (no timeperiod where there is no event but there is an earlier and later event on the same day) on every given time group (day) (soft)
28) MinMaxADay_Classes1: [Individual Classes] Classes should attend between 1 and 4 periods every day (soft)
29) MinMaxADay_Classes2: [Individual Classes] Classes should attend between 1 and 4 periods every day (soft)
30) MinMaxADay_Teachers1: [Individual Teachers] Teachers should teach between 1 and 4 periods every day (soft)
31) MinMaxADay_Teachers2: [Individual Teachers] Teachers should teach between 2 and 4 periods every day (soft)



Part 3: Event -> room:

1) Assign Room X: [gr_EventsGeneral(gr_Room_X)] events with role gr_Room_X must be allocated to any room.
2) Assign Room Y: [gr_EventsGeneral(gr_Room_Y)] ...
3) Assign Room Z: [gr_EventsGeneral(gr_Room_Z)] ...
6) PreferredResourcesXRooms: [gr_EventsGeneral(gr_Room_X)] events with role gr_Room_X must be allocated to a room in gr_Room_X 
7) PreferredResourcesYRooms: [gr_EventsGeneral(gr_Room_Y)] ...
8) PreferredResourcesZRooms: [gr_EventsGeneral(gr_Room_Z)] ...

### MODEL

In [99]:
# ---- Model ----
m1 = pyo.ConcreteModel()

# ---- Sets ----
m1.E = pyo.Set(initialize=E)
m1.T = pyo.Set(initialize=T, ordered=True)
m1.R = pyo.Set(initialize=R)

m1.Teachers = pyo.Set(initialize=all_teachers)
m1.Classes = pyo.Set(initialize=all_classes)

# feasible (event, start) pairs
m1.ES_index = pyo.Set(
    dimen=2,
    initialize=[(e, ts) for e in E for ts in feasible_starts[e]]
)

# feasible (event, start, room) triples
m1.X_index = pyo.Set(
    dimen=3,
    initialize=[
        (e, ts, r)
        for e in E
        for ts in feasible_starts[e]
        for r in feasible_rooms[e]
    ]
)

# Map (time, room) -> feasible triples occupying that room at that time
occupies_at_time_room = {(t, r): [] for t in T for r in R}
for (e, ts, r) in m1.X_index:
    for t in occ_times[(e, ts)]:
        occupies_at_time_room[(t, r)].append((e, ts, r))

# ---- Decision Variables ----
m1.x = pyo.Var(m1.X_index, domain=pyo.Binary)

# helper: 1 if event e starts at ts in some feasible room
def start_sum_rule(m, e, ts):
    return sum(m.x[e, ts, r] for r in feasible_rooms[e] if (e, ts, r) in m.X_index)

m1.start_sum = pyo.Expression(m1.ES_index, rule=start_sum_rule)


In [100]:
# Standard set up

# 1) Each event scheduled exactly once
def one_start_rule(m, e):
    return sum(
        m.x[e, ts, r]
        for ts in feasible_starts[e]
        for r in feasible_rooms[e]
        if (e, ts, r) in m.X_index
    ) == 1
m1.OneStart = pyo.Constraint(m1.E, rule=one_start_rule)

# 2) No room conflicts
def room_conflict_rule(m, r, t):
    terms = [m.x[e, ts, rr] for (e, ts, rr) in occupies_at_time_room[(t, r)]]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1
m1.RoomConflict = pyo.Constraint(m1.R, m1.T, rule=room_conflict_rule)

# 3) No teacher conflicts
def teacher_conflict_rule(m, teacher, t):
    terms = [
        m.start_sum[e, ts]
        for (e, ts) in occupies_at_time.get(t, [])
        if teacher in as_list(event_to_teachers[e])
    ]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1
m1.TeacherConflict = pyo.Constraint(m1.Teachers, m1.T, rule=teacher_conflict_rule)

# 4) No class conflicts
def class_conflict_rule(m, cls, t):
    terms = [
        m.start_sum[e, ts]
        for (e, ts) in occupies_at_time.get(t, [])
        if cls in event_to_classes[e]
    ]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1
m1.ClassConflict = pyo.Constraint(m1.Classes, m1.T, rule=class_conflict_rule)

# 5) Room-group capacities
#roomgroup_cap = {
#    "gr_Room_X": 1,
#    "gr_Room_Y": 2,
#    "gr_Room_Z": 9,
#}

#m1.G = pyo.Set(initialize=roomgroup_cap.keys())


#m1.RoomGroupCapacity = pyo.Constraint(m1.G, m1.T, rule=roomgroup_capacity_rule)


In [101]:
### 3) Events that are part of the same course must be on different days (constraint 13)

event_to_course = dict(zip(df_events["event_id"], df_events["course_reference"]))
courses = sorted(set(event_to_course[e] for e in E))
days = sorted(df_times["day_ref"].dropna().unique().tolist())

course_day_terms = defaultdict(list)
for (e, ts) in m1.ES_index:
    c = event_to_course[e]
    d = time_to_day[ts]
    course_day_terms[(c, d)].append((e, ts))

m1.COURSES = pyo.Set(initialize=courses)
m1.DAYS = pyo.Set(initialize=days)

def event_spread_max_rule(m, c, d):
    terms = course_day_terms.get((c, d), [])
    if not terms:
        return pyo.Constraint.Skip
    return sum(m.start_sum[e, ts] for (e, ts) in terms) <= 1

m1.EventSpreadMax = pyo.Constraint(m1.COURSES, m1.DAYS, rule=event_spread_max_rule)


In [78]:
#### FOR EXPLICIT FEASIBLE STARTS / ROOMS WITH x[e, ts, r]

# Allowed
role_to_roomgroup = {
    "gr_Room_X": "gr_Rooms_X",
    "gr_Room_Y": "gr_Rooms_Y",
    "gr_Room_Z": "gr_Rooms_Z",
}

allowed_rooms_by_event = {}
for e in E:
    role = event_to_roomgroup[e]               # e -> gr_Room_X/Y/Z
    rg = role_to_roomgroup[role]               # -> gr_Rooms_X/Y/Z
    allowed_rooms_by_event[e] = group_to_rooms[rg]


# Same
start_same_day_ok = {}
for e in E:
    dur = int(event_to_duration[e])
    for ts in T:
        i0 = T_index[ts]
        if i0 + dur - 1 >= len(T):
            ok = 0
        else:
            block = T[i0:i0 + dur]
            ok = int(all(time_to_day[t] == time_to_day[ts] for t in block))

        for r in R:
            start_same_day_ok[(e, ts, r)] = ok

m1.start_same_day_ok = pyo.Param(m1.X_index, initialize=start_same_day_ok, within=pyo.Binary)

def same_day_start_rule(m, e, ts, r):
    return m.x[e, ts, r] <= m.start_same_day_ok[e, ts, r]

m1.SameDayStart = pyo.Constraint(m1.X_index, rule=same_day_start_rule)


# Teacher
start_teacher_ok = {}
for e in E:
    teachers_e = set(as_list(event_to_teachers[e]))

    for ts in T:
        ok = 1

        if (e, ts) not in occ_times:
            ok = 0
        else:
            block = occ_times[(e, ts)]
            for info in unavailability.values():
                teacher = info["teacher"]
                if teacher in teachers_e:
                    unavail_set = set(info["times"])
                    if any(t in unavail_set for t in block):
                        ok = 0
                        break

        for r in R:
            start_teacher_ok[(e, ts, r)] = ok

m1.start_teacher_ok = pyo.Param(m1.X_index, initialize=start_teacher_ok, within=pyo.Binary)

def teacher_availability_start_rule(m, e, ts, r):
    return m.x[e, ts, r] <= m.start_teacher_ok[e, ts, r]

m1.TeacherStartAllowed = pyo.Constraint(m1.X_index, rule=teacher_availability_start_rule)


# Allowed
room_group_ok = {}
for e in E:
    allowed_set = set(allowed_rooms_by_event[e])
    for ts in T:
        for r in R:
            room_group_ok[(e, ts, r)] = int(r in allowed_set)

m1.room_group_ok = pyo.Param(m1.X_index, initialize=room_group_ok, within=pyo.Binary)

def allowed_room_rule(m, e, ts, r):
    return m.x[e, ts, r] <= m.room_group_ok[e, ts, r]

m1.AllowedRoomGroup = pyo.Constraint(m1.X_index, rule=allowed_room_rule)


In [84]:
#### COMBINED EXPLICIT START/ROOM ELIGIBILITY FOR x[e, ts, r]

# Allowed
role_to_roomgroup = {
    "gr_Room_X": "gr_Rooms_X",
    "gr_Room_Y": "gr_Rooms_Y",
    "gr_Room_Z": "gr_Rooms_Z",
}

allowed_rooms_by_event = {}
for e in E:
    role = event_to_roomgroup[e]
    rg = role_to_roomgroup[role]
    allowed_rooms_by_event[e] = group_to_rooms[rg]


# Combined
start_assignment_ok = {}

for e in E:
    dur = int(event_to_duration[e])
    teachers_e = set(as_list(event_to_teachers[e]))
    allowed_room_set = set(allowed_rooms_by_event[e])

    for ts in T:
        i0 = T_index[ts]

        # same-day check
        if i0 + dur - 1 >= len(T):
            same_day_ok = 0
            block = []
        else:
            block = T[i0:i0 + dur]
            same_day_ok = int(all(time_to_day[t] == time_to_day[ts] for t in block))

        # teacher availability check
        teacher_ok = 1
        if not block:
            teacher_ok = 0
        else:
            for info in unavailability.values():
                teacher = info["teacher"]
                if teacher in teachers_e:
                    unavail_set = set(info["times"])
                    if any(t in unavail_set for t in block):
                        teacher_ok = 0
                        break

        for r in R:
            room_ok = int(r in allowed_room_set)
            start_assignment_ok[(e, ts, r)] = same_day_ok * teacher_ok * room_ok


m1.start_assignment_ok = pyo.Param(
    m1.X_index,
    initialize=start_assignment_ok,
    within=pyo.Binary
)

def start_assignment_allowed_rule(m, e, ts, r):
    return m.x[e, ts, r] <= m.start_assignment_ok[e, ts, r]

m1.StartAssignmentAllowed = pyo.Constraint(
    m1.X_index,
    rule=start_assignment_allowed_rule
)


In [102]:
# ---- Objective ---- (hard constraints only)
m1.obj = pyo.Objective(expr=0, sense=pyo.minimize)

In [103]:
import numpy as np

In [104]:
### Preprocessing
seeds = [1, 2, 5, 7, 10]

solve_times = []
objectives = []
num_vars = []
num_cons = []
solve_status = []
start_x_list = []

import time as time_module

def clear_all_var_values(model):
    for v in model.component_data_objects(pyo.Var, active=True):
        v.set_value(None, skip_validation=True)

for seed in seeds:
    clear_all_var_values(m1)

    solver = pyo.SolverFactory("gurobi")
    #solver.options["TimeLimit"] = 600
    solver.options["Seed"] = seed
    solver.options["Threads"] = 4

    # Manual timing
    start_time = time_module.time()
    res = solver.solve(m1, tee=False)
    elapsed_time = time_module.time() - start_time

    solve_times.append(elapsed_time)
    objectives.append(pyo.value(m1.obj))
    num_vars.append(m1.nvariables())
    num_cons.append(m1.nconstraints())

    solve_status.append(res.solver.termination_condition)

    # Store solution
    # Store solution
    seed_solution = {}
    for (e, ts, r) in m1.X_index:
        val = pyo.value(m1.x[e, ts, r], exception=False)
        seed_solution[(e, ts, r)] = 0.0 if val is None else float(val)
    start_x_list.append(seed_solution)

# Keep compatibility with later cells that expect start_x / res
start_x = start_x_list[0]
res = res

# Calculate averages
avg_time = sum(solve_times) / len(solve_times)
avg_obj = sum(objectives) / len(objectives)
avg_vars = sum(num_vars) / len(num_vars)
avg_cons = sum(num_cons) / len(num_cons)

print("=" * 50)
print("SOLVER METRICS SUMMARY - PHASE 1")
print("=" * 50)
print(f"Average Solve Time (s): {avg_time:.4f}")
print(f"Average Objective: {avg_obj:.6f}")
print(f"Average Variables: {avg_vars:.0f}")
print(f"Average Constraints: {avg_cons:.0f}")
print(f"Termination Status: {set(solve_status)}")
print("=" * 50)

print(f"\nPhase 1 produced {len(start_x_list)} solutions (one per seed)")
for i, seed_sol in enumerate(start_x_list):
    scheduled = sum(1 for v in seed_sol.values() if v > 0)
    print(f"  Seed {seeds[i]}: {scheduled} events scheduled in {solve_times[i]:.2f}s, Obj={objectives[i]:.6f}")


SOLVER METRICS SUMMARY - PHASE 1
Average Solve Time (s): 173.7015
Average Objective: 0.000000
Average Variables: 17364
Average Constraints: 1616
Termination Status: {<TerminationCondition.optimal: 'optimal'>}

Phase 1 produced 5 solutions (one per seed)
  Seed 1: 169 events scheduled in 161.60s, Obj=0.000000
  Seed 2: 169 events scheduled in 194.10s, Obj=0.000000
  Seed 5: 169 events scheduled in 175.58s, Obj=0.000000
  Seed 7: 169 events scheduled in 155.07s, Obj=0.000000
  Seed 10: 169 events scheduled in 182.15s, Obj=0.000000


In [80]:
### Explicit 1
seeds = [1, 2, 5, 7, 10]

solve_times = []
objectives = []
num_vars = []
num_cons = []
solve_status = []
start_x_list = []

import time as time_module

def clear_all_var_values(model):
    for v in model.component_data_objects(pyo.Var, active=True):
        v.set_value(None, skip_validation=True)

for seed in seeds:
    clear_all_var_values(m1)

    solver = pyo.SolverFactory("gurobi")
    # solver.options["TimeLimit"] = 600
    solver.options["Seed"] = seed
    solver.options["Threads"] = 1

    start_time = time_module.time()
    res = solver.solve(m1, tee=False, load_solutions=False)
    elapsed_time = time_module.time() - start_time

    status = str(res.solver.status)
    term = str(res.solver.termination_condition)
    has_solution = len(res.solution) > 0

    if has_solution:
        m1.solutions.load_from(res)

    solve_times.append(elapsed_time)
    num_vars.append(m1.nvariables())
    num_cons.append(m1.nconstraints())
    solve_status.append((status, term))

    if has_solution:
        try:
            obj_val = pyo.value(m1.obj)
        except:
            obj_val = float("nan")
            has_solution = False
    else:
        obj_val = float("nan")

    objectives.append(obj_val)

    if has_solution:
        seed_solution = {}
        for (e, ts, r) in m1.X_index:
            val = pyo.value(m1.x[e, ts, r], exception=False)
            seed_solution[(e, ts, r)] = 0.0 if val is None else float(val)
        start_x_list.append(seed_solution)

if start_x_list:
    start_x = start_x_list[0]
else:
    start_x = {}

res = res

avg_time = sum(solve_times) / len(solve_times) if solve_times else float("nan")
avg_vars = sum(num_vars) / len(num_vars) if num_vars else float("nan")
avg_cons = sum(num_cons) / len(num_cons) if num_cons else float("nan")
valid_objs = [v for v in objectives if not np.isnan(v)]

print("=" * 50)
print("SOLVER METRICS SUMMARY - PHASE 1")
print("=" * 50)
print(f"Average Solve Time (s): {avg_time:.4f}" if solve_times else "Average Solve Time (s): Not available")
print(f"Average Objective: {np.mean(valid_objs):.6f}" if valid_objs else "Average Objective: Not available")
print(f"Average Variables: {avg_vars:.0f}" if num_vars else "Average Variables: Not available")
print(f"Average Constraints: {avg_cons:.0f}" if num_cons else "Average Constraints: Not available")
print(f"Statuses: {set(solve_status)}")
print("=" * 50)

print(f"\nPhase 1 produced {len(start_x_list)} solutions (one per seed)")
for i in range(len(seeds)):
    obj_str = f"{objectives[i]:.6f}" if not np.isnan(objectives[i]) else "No solution"
    print(
        f"  Seed {seeds[i]}: Time={solve_times[i]:.2f}s, "
        f"Obj={obj_str}, Status={solve_status[i]}"
    )


SOLVER METRICS SUMMARY - PHASE 1
Average Solve Time (s): 520.5959
Average Objective: 0.000000
Average Variables: 40560
Average Constraints: 123539
Statuses: {('ok', 'optimal')}

Phase 1 produced 5 solutions (one per seed)
  Seed 1: Time=102.69s, Obj=0.000000, Status=('ok', 'optimal')
  Seed 2: Time=978.68s, Obj=0.000000, Status=('ok', 'optimal')
  Seed 5: Time=880.43s, Obj=0.000000, Status=('ok', 'optimal')
  Seed 7: Time=140.49s, Obj=0.000000, Status=('ok', 'optimal')
  Seed 10: Time=500.69s, Obj=0.000000, Status=('ok', 'optimal')


In [86]:
### Explicit 2
import time as time_module

seeds = [1, 2, 5, 7, 10]

solve_times = []
objectives = []
num_vars = []
num_cons = []
solve_status = []
start_x_list = []

def clear_all_var_values(model):
    for v in model.component_data_objects(pyo.Var, active=True):
        v.set_value(None, skip_validation=True)

for seed in seeds:
    clear_all_var_values(m1)

    solver = pyo.SolverFactory("gurobi")
    # solver.options["TimeLimit"] = 600
    solver.options["Seed"] = seed
    solver.options["Threads"] = 4

    start_time = time_module.time()
    res = solver.solve(m1, tee=False, load_solutions=False)
    elapsed_time = time_module.time() - start_time

    status = str(res.solver.status)
    term = str(res.solver.termination_condition)
    has_solution = len(res.solution) > 0

    if has_solution:
        m1.solutions.load_from(res)

    solve_times.append(elapsed_time)
    num_vars.append(m1.nvariables())
    num_cons.append(m1.nconstraints())
    solve_status.append((status, term))

    if has_solution:
        try:
            obj_val = pyo.value(m1.obj)
        except:
            obj_val = float("nan")
            has_solution = False
    else:
        obj_val = float("nan")

    objectives.append(obj_val)

    seed_solution = {}
    if has_solution:
        for (e, ts, r) in m1.X_index:
            val = pyo.value(m1.x[e, ts, r], exception=False)
            seed_solution[(e, ts, r)] = 0.0 if val is None else float(val)

    start_x_list.append(seed_solution)

# Keep compatibility with later cells that expect start_x / res
start_x = start_x_list[0] if start_x_list else {}
res = res

# Summary statistics
avg_time = sum(solve_times) / len(solve_times) if solve_times else float("nan")
avg_vars = sum(num_vars) / len(num_vars) if num_vars else float("nan")
avg_cons = sum(num_cons) / len(num_cons) if num_cons else float("nan")
valid_objs = [v for v in objectives if not np.isnan(v)]

print("=" * 50)
print("SOLVER METRICS SUMMARY - PHASE 1")
print("=" * 50)
print(f"Average Solve Time (s): {avg_time:.4f}" if solve_times else "Average Solve Time (s): Not available")
print(f"Average Objective: {np.mean(valid_objs):.6f}" if valid_objs else "Average Objective: Not available")
print(f"Average Variables: {avg_vars:.0f}" if num_vars else "Average Variables: Not available")
print(f"Average Constraints: {avg_cons:.0f}" if num_cons else "Average Constraints: Not available")
print(f"Statuses: {set(solve_status)}")
print("=" * 50)

print(f"\nPhase 1 produced {len(start_x_list)} solutions (one per seed)")
for i, seed_sol in enumerate(start_x_list):
    if seed_sol:
        scheduled = sum(1 for v in seed_sol.values() if v > 0)
        obj_str = f"{objectives[i]:.6f}" if not np.isnan(objectives[i]) else "No solution"
        print(
            f"  Seed {seeds[i]}: Time={solve_times[i]:.2f}s, "
            f"Obj={obj_str}, Scheduled={scheduled}, Status={solve_status[i]}"
        )
    else:
        print(
            f"  Seed {seeds[i]}: Time={solve_times[i]:.2f}s, "
            f"Obj=No solution, Scheduled=0, Status={solve_status[i]}"
        )


## 1 and 5 and maybe 2
## 10 maybe 6 7

SOLVER METRICS SUMMARY - PHASE 1
Average Solve Time (s): 171.7448
Average Objective: 0.000000
Average Variables: 40560
Average Constraints: 42419
Statuses: {('ok', 'optimal')}

Phase 1 produced 5 solutions (one per seed)
  Seed 1: Time=154.12s, Obj=0.000000, Scheduled=169, Status=('ok', 'optimal')
  Seed 2: Time=191.15s, Obj=0.000000, Scheduled=169, Status=('ok', 'optimal')
  Seed 5: Time=197.53s, Obj=0.000000, Scheduled=169, Status=('ok', 'optimal')
  Seed 7: Time=157.89s, Obj=0.000000, Scheduled=169, Status=('ok', 'optimal')
  Seed 10: Time=158.04s, Obj=0.000000, Scheduled=169, Status=('ok', 'optimal')


### HYBRID

In [87]:
### Occupancy - Start Time Hybrid

from collections import defaultdict
import pyomo.environ as pyo
import numpy as np
import time as time_module

# Build
occ_times = {}
covering_starts = defaultdict(list)       # (e, t) -> [(ts, r), ...]
occupies_at_time_room = defaultdict(list) # (t, r) -> [(e, ts), ...]

for e in E:
    dur = int(event_to_duration[e])
    for ts in feasible_starts[e]:
        i0 = T_index[ts]
        block = T[i0:i0 + dur]
        occ_times[(e, ts)] = block

        for r in feasible_rooms[e]:
            for t in block:
                covering_starts[(e, t)].append((ts, r))
                occupies_at_time_room[(t, r)].append((e, ts))

# Model
mb = pyo.ConcreteModel()

mb.E = pyo.Set(initialize=E)
mb.T = pyo.Set(initialize=T, ordered=True)
mb.R = pyo.Set(initialize=R)

mb.X_index = pyo.Set(
    dimen=3,
    initialize=[
        (e, ts, r)
        for e in E
        for ts in feasible_starts[e]
        for r in feasible_rooms[e]
    ]
)

# Start-time-and-room decision
mb.x = pyo.Var(mb.X_index, domain=pyo.Binary)

# Occupancy indicator by event and time
mb.w = pyo.Var(mb.E, mb.T, domain=pyo.Binary)

# Explicit
start_allowed = {}

for e in E:
    dur = int(event_to_duration[e])
    teachers_e = set(as_list(event_to_teachers[e]))

    for ts in feasible_starts[e]:
        i0 = T_index[ts]

        if i0 + dur - 1 >= len(T):
            allowed = 0
        else:
            block = T[i0:i0 + dur]
            day0 = time_to_day[ts]

            if not all(time_to_day[t] == day0 for t in block):
                allowed = 0
            else:
                violates_unavailability = False
                for info in unavailability.values():
                    teacher = info["teacher"]
                    if teacher in teachers_e:
                        unavail_set = set(info["times"])
                        if any(t in unavail_set for t in block):
                            violates_unavailability = True
                            break
                allowed = 0 if violates_unavailability else 1

        for r in feasible_rooms[e]:
            start_allowed[(e, ts, r)] = allowed

mb.start_allowed = pyo.Param(
    mb.X_index,
    initialize=start_allowed,
    within=pyo.Binary
)

def valid_start_rule(mb, e, ts, r):
    return mb.x[e, ts, r] <= mb.start_allowed[e, ts, r]

mb.ValidStart = pyo.Constraint(mb.X_index, rule=valid_start_rule)

# Each
def one_start_rule(mb, e):
    return sum(
        mb.x[e, ts, r]
        for ts in feasible_starts[e]
        for r in feasible_rooms[e]
    ) == 1

mb.OneStart = pyo.Constraint(mb.E, rule=one_start_rule)

# Link
def occupancy_link_rule(mb, e, t):
    starts_that_cover_t = covering_starts.get((e, t), [])
    if not starts_that_cover_t:
        return mb.w[e, t] == 0
    return mb.w[e, t] == sum(mb.x[e, ts, r] for (ts, r) in starts_that_cover_t)

mb.OccupancyLink = pyo.Constraint(mb.E, mb.T, rule=occupancy_link_rule)

# No
def room_conflict_rule(mb, r, t):
    terms = [
        mb.x[e, ts, r]
        for (e, ts) in occupies_at_time_room.get((t, r), [])
        if (e, ts, r) in mb.X_index
    ]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1

mb.RoomConflict = pyo.Constraint(mb.R, mb.T, rule=room_conflict_rule)

# No
def teacher_clash_rule(mb, teacher, t):
    terms = [mb.w[e, t] for e in mb.E if teacher in as_list(event_to_teachers[e])]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1

mb.TeacherClash = pyo.Constraint(all_teachers, mb.T, rule=teacher_clash_rule)

# No
def class_clash_rule(mb, cls, t):
    terms = [mb.w[e, t] for e in mb.E if cls in as_list(event_to_classes[e])]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1

mb.ClassClash = pyo.Constraint(all_classes, mb.T, rule=class_clash_rule)

# Events
event_to_course = dict(zip(df_events["event_id"], df_events["course_reference"]))
courses = sorted(set(event_to_course[e] for e in E))
days = sorted(df_times["day_ref"].dropna().unique().tolist())

course_day_terms = defaultdict(list)
for (e, ts, r) in mb.X_index:
    c = event_to_course[e]
    d = time_to_day[ts]
    course_day_terms[(c, d)].append((e, ts, r))

mb.COURSES = pyo.Set(initialize=courses)
mb.DAYS = pyo.Set(initialize=days)

def event_spread_max_rule(mb, c, d):
    terms = course_day_terms.get((c, d), [])
    if not terms:
        return pyo.Constraint.Skip
    return sum(mb.x[e, ts, r] for (e, ts, r) in terms) <= 1

mb.EventSpreadMax = pyo.Constraint(mb.COURSES, mb.DAYS, rule=event_spread_max_rule)

# Objective
mb.obj = pyo.Objective(expr=0, sense=pyo.minimize)


In [88]:
### Explicit 2
seeds = [1, 2, 5, 7, 10]

solve_times = []
objectives = []
num_vars = []
num_cons = []
solve_status = []
start_x_list = []

def clear_all_var_values(model):
    for v in model.component_data_objects(pyo.Var, active=True):
        v.set_value(None, skip_validation=True)

for seed in seeds:
    clear_all_var_values(mb)

    solver = pyo.SolverFactory("gurobi")
    # solver.options["TimeLimit"] = 600
    solver.options["Seed"] = seed
    solver.options["Threads"] = 4

    start_time = time_module.time()
    res = solver.solve(mb, tee=False, load_solutions=False)
    elapsed_time = time_module.time() - start_time

    status = str(res.solver.status)
    term = str(res.solver.termination_condition)
    has_solution = len(res.solution) > 0

    if has_solution:
        mb.solutions.load_from(res)

    solve_times.append(elapsed_time)
    num_vars.append(mb.nvariables())
    num_cons.append(mb.nconstraints())
    solve_status.append((status, term))

    if has_solution:
        try:
            obj_val = pyo.value(mb.obj)
        except:
            obj_val = float("nan")
            has_solution = False
    else:
        obj_val = float("nan")

    objectives.append(obj_val)

    seed_solution = {}
    if has_solution:
        for (e, ts, r) in mb.X_index:
            val = pyo.value(mb.x[e, ts, r], exception=False)
            seed_solution[(e, ts, r)] = 0.0 if val is None else float(val)

    start_x_list.append(seed_solution)

start_x = start_x_list[0] if start_x_list else {}
res = res

avg_time = sum(solve_times) / len(solve_times) if solve_times else float("nan")
avg_vars = sum(num_vars) / len(num_vars) if num_vars else float("nan")
avg_cons = sum(num_cons) / len(num_cons) if num_cons else float("nan")
valid_objs = [v for v in objectives if not np.isnan(v)]

print("=" * 50)
print("SOLVER METRICS SUMMARY - PHASE 1")
print("=" * 50)
print(f"Average Solve Time (s): {avg_time:.4f}" if solve_times else "Average Solve Time (s): Not available")
print(f"Average Objective: {np.mean(valid_objs):.6f}" if valid_objs else "Average Objective: Not available")
print(f"Average Variables: {avg_vars:.0f}" if num_vars else "Average Variables: Not available")
print(f"Average Constraints: {avg_cons:.0f}" if num_cons else "Average Constraints: Not available")
print(f"Statuses: {set(solve_status)}")
print("=" * 50)

print(f"\nPhase 1 produced {len(start_x_list)} solutions (one per seed)")
for i, seed_sol in enumerate(start_x_list):
    assigned = sum(1 for v in seed_sol.values() if v > 0.5)
    obj_str = f"{objectives[i]:.6f}" if not np.isnan(objectives[i]) else "No solution"
    print(
        f"  Seed {seeds[i]}: Time={solve_times[i]:.2f}s, "
        f"Obj={obj_str}, Assigned triples={assigned}, Status={solve_status[i]}"
    )


SOLVER METRICS SUMMARY - PHASE 1
Average Solve Time (s): 173.3551
Average Objective: 0.000000
Average Variables: 43940
Average Constraints: 45799
Statuses: {('ok', 'optimal')}

Phase 1 produced 5 solutions (one per seed)
  Seed 1: Time=258.75s, Obj=0.000000, Assigned triples=169, Status=('ok', 'optimal')
  Seed 2: Time=115.70s, Obj=0.000000, Assigned triples=169, Status=('ok', 'optimal')
  Seed 5: Time=214.09s, Obj=0.000000, Assigned triples=169, Status=('ok', 'optimal')
  Seed 7: Time=105.15s, Obj=0.000000, Assigned triples=169, Status=('ok', 'optimal')
  Seed 10: Time=173.08s, Obj=0.000000, Assigned triples=169, Status=('ok', 'optimal')


### OCCUPANCY

In [89]:
### Pure Occupancy

times_by_day = {}
for _, row in df_times.sort_values("order_index").iterrows():
    d = row["day_ref"]
    times_by_day.setdefault(d, []).append(row["time_id"])

days = list(times_by_day.keys())

prev_in_day = {}
next_in_day = {}

for d in days:
    day_times = times_by_day[d]
    for i, t in enumerate(day_times):
        prev_in_day[t] = None if i == 0 else day_times[i - 1]
        next_in_day[t] = None if i == len(day_times) - 1 else day_times[i + 1]

# Build
illegal_starts = []

for e in E:
    dur = int(event_to_duration[e])
    teachers_e = set(as_list(event_to_teachers[e]))

    for ts in T:
        i0 = T_index[ts]

        if i0 + dur - 1 >= len(T):
            continue

        block = T[i0:i0 + dur]
        day0 = time_to_day[ts]

        same_day = all(time_to_day[t] == day0 for t in block)

        violates_unavailability = False
        for info in unavailability.values():
            teacher = info["teacher"]
            if teacher in teachers_e:
                unavail_set = set(info["times"])
                if any(t in unavail_set for t in block):
                    violates_unavailability = True
                    break

        if (not same_day) or violates_unavailability:
            illegal_starts.append((e, ts))


In [90]:
mc = pyo.ConcreteModel()

mc.E = pyo.Set(initialize=E)
mc.T = pyo.Set(initialize=T, ordered=True)
mc.D = pyo.Set(initialize=days)
mc.R = pyo.Set(initialize=R)

# Event occupancy by time
mc.w = pyo.Var(mc.E, mc.T, domain=pyo.Binary)

# Room assigned to event
mc.Q_index = pyo.Set(
    dimen=2,
    initialize=[(e, r) for e in E for r in feasible_rooms[e]]
)
mc.q = pyo.Var(mc.Q_index, domain=pyo.Binary)

# Event occupies room r at time t
mc.Z_index = pyo.Set(
    dimen=3,
    initialize=[(e, t, r) for e in E for t in T for r in feasible_rooms[e]]
)
mc.z = pyo.Var(mc.Z_index, domain=pyo.Binary)

mc.U_index = pyo.Set(
    dimen=3,
    initialize=[(e, d, t) for e in E for d in days for t in times_by_day[d]]
)
mc.u = pyo.Var(mc.U_index, domain=pyo.Binary)
mc.v = pyo.Var(mc.U_index, domain=pyo.Binary)


In [ ]:
def duration_rule(mc, e):
    return sum(mc.w[e, t] for t in mc.T) == int(event_to_duration[e])
mc.Duration = pyo.Constraint(mc.E, rule=duration_rule)

mc.ILLEGAL_STARTS = pyo.Set(dimen=2, initialize=illegal_starts)

def illegal_block_rule(mc, e, ts):
    dur = int(event_to_duration[e])
    i0 = T_index[ts]
    block = T[i0:i0 + dur]
    return sum(mc.w[e, t] for t in block) <= dur - 1
mc.IllegalBlock = pyo.Constraint(mc.ILLEGAL_STARTS, rule=illegal_block_rule)

# Each event chooses exactly one feasible room
def one_room_rule(mc, e):
    return sum(mc.q[e, r] for r in feasible_rooms[e]) == 1
mc.OneRoom = pyo.Constraint(mc.E, rule=one_room_rule)

# Link event occupancy and chosen room to room-time occupancy
def room_occ_upper_time_rule(mc, e, t, r):
    return mc.z[e, t, r] <= mc.w[e, t]
mc.RoomOccUpperTime = pyo.Constraint(mc.Z_index, rule=room_occ_upper_time_rule)

def room_occ_upper_room_rule(mc, e, t, r):
    return mc.z[e, t, r] <= mc.q[e, r]
mc.RoomOccUpperRoom = pyo.Constraint(mc.Z_index, rule=room_occ_upper_room_rule)

def room_occ_lower_rule(mc, e, t, r):
    return mc.z[e, t, r] >= mc.w[e, t] + mc.q[e, r] - 1
mc.RoomOccLower = pyo.Constraint(mc.Z_index, rule=room_occ_lower_rule)

# No room clashes
def room_conflict_rule(mc, r, t):
    terms = [mc.z[e, t, r] for e in mc.E if (e, t, r) in mc.Z_index]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1
mc.RoomConflict = pyo.Constraint(mc.R, mc.T, rule=room_conflict_rule)

# No teacher clashes
def teacher_clash_rule(mc, teacher, t):
    terms = [mc.w[e, t] for e in mc.E if teacher in as_list(event_to_teachers[e])]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1
mc.TeacherClash = pyo.Constraint(all_teachers, mc.T, rule=teacher_clash_rule)

# No class clashes
def class_clash_rule(mc, cls, t):
    terms = [mc.w[e, t] for e in mc.E if cls in as_list(event_to_classes[e])]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1
mc.ClassClash = pyo.Constraint(all_classes, mc.T, rule=class_clash_rule)


In [ ]:
# Mark the first occupied period of an event on a day
def start_edge_rule(mc, e, d, t):
    p = prev_in_day[t]
    if p is None:
        return mc.u[e, d, t] >= mc.w[e, t]
    return mc.u[e, d, t] >= mc.w[e, t] - mc.w[e, p]
mc.StartEdge = pyo.Constraint(mc.U_index, rule=start_edge_rule)

# Mark the last occupied period of an event on a day
def end_edge_rule(mc, e, d, t):
    n = next_in_day[t]
    if n is None:
        return mc.v[e, d, t] >= mc.w[e, t]
    return mc.v[e, d, t] >= mc.w[e, t] - mc.w[e, n]
mc.EndEdge = pyo.Constraint(mc.U_index, rule=end_edge_rule)

# Allow at most one daily start marker per event
def one_start_per_day_rule(mc, e, d):
    return sum(mc.u[e, d, t] for t in times_by_day[d]) <= 1
mc.OneStartPerDay = pyo.Constraint(mc.E, mc.D, rule=one_start_per_day_rule)

# Allow at most one daily end marker per event
def one_end_per_day_rule(mc, e, d):
    return sum(mc.v[e, d, t] for t in times_by_day[d]) <= 1
mc.OneEndPerDay = pyo.Constraint(mc.E, mc.D, rule=one_end_per_day_rule)

# Force exactly one start marker across the full schedule
def one_start_overall_rule(mc, e):
    return sum(mc.u[e, d, t] for d in mc.D for t in times_by_day[d]) == 1
mc.OneStartOverall = pyo.Constraint(mc.E, rule=one_start_overall_rule)

# Force exactly one end marker across the full schedule
def one_end_overall_rule(mc, e):
    return sum(mc.v[e, d, t] for d in mc.D for t in times_by_day[d]) == 1
mc.OneEndOverall = pyo.Constraint(mc.E, rule=one_end_overall_rule)


In [93]:
### 3) Events that are part of the same course must be on different days (constraint 13)

mc.day_used = pyo.Var(mc.E, mc.D, domain=pyo.Binary)

def day_used_upper_rule(mc, e, d):
    return sum(mc.w[e, t] for t in times_by_day[d]) <= len(times_by_day[d]) * mc.day_used[e, d]
mc.DayUsedUpper = pyo.Constraint(mc.E, mc.D, rule=day_used_upper_rule)

def day_used_lower_rule(mc, e, d):
    return sum(mc.w[e, t] for t in times_by_day[d]) >= mc.day_used[e, d]
mc.DayUsedLower = pyo.Constraint(mc.E, mc.D, rule=day_used_lower_rule)

event_to_course = dict(zip(df_events["event_id"], df_events["course_reference"]))
courses = sorted(set(event_to_course[e] for e in E))

mc.COURSES = pyo.Set(initialize=courses)

def event_spread_max_rule(mc, c, d):
    terms = [mc.day_used[e, d] for e in mc.E if event_to_course[e] == c]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1
mc.EventSpreadMax = pyo.Constraint(mc.COURSES, mc.D, rule=event_spread_max_rule)

mc.obj = pyo.Objective(expr=0, sense=pyo.minimize)


In [94]:
### Full Occupancy
seeds = [1, 2, 5, 7, 10]

solve_times = []
objectives = []
num_vars = []
num_cons = []
solve_status = []
start_x_list = []

import time as time_module

def clear_all_var_values(model):
    for v in model.component_data_objects(pyo.Var, active=True):
        v.set_value(None, skip_validation=True)

for seed in seeds:
    clear_all_var_values(mc)

    solver = pyo.SolverFactory("gurobi")
    # solver.options["TimeLimit"] = 600
    solver.options["Seed"] = seed
    solver.options["Threads"] = 4

    start_time = time_module.time()
    res = solver.solve(mc, tee=False, load_solutions=False)
    elapsed_time = time_module.time() - start_time

    status = str(res.solver.status)
    term = str(res.solver.termination_condition)
    has_solution = len(res.solution) > 0

    if has_solution:
        mc.solutions.load_from(res)

    solve_times.append(elapsed_time)
    num_vars.append(mc.nvariables())
    num_cons.append(mc.nconstraints())
    solve_status.append((status, term))

    if has_solution:
        try:
            obj_val = pyo.value(mc.obj)
        except:
            obj_val = float("nan")
            has_solution = False
    else:
        obj_val = float("nan")

    objectives.append(obj_val)

    seed_solution = {}
    if has_solution:
        for (e, t, r) in mc.Z_index:
            val = pyo.value(mc.z[e, t, r], exception=False)
            seed_solution[(e, t, r)] = 0.0 if val is None else float(val)

    start_x_list.append(seed_solution)

start_x = start_x_list[0] if start_x_list else {}
res = res

avg_time = sum(solve_times) / len(solve_times) if solve_times else float("nan")
avg_vars = sum(num_vars) / len(num_vars) if num_vars else float("nan")
avg_cons = sum(num_cons) / len(num_cons) if num_cons else float("nan")
valid_objs = [v for v in objectives if not np.isnan(v)]

print("=" * 50)
print("SOLVER METRICS SUMMARY - PHASE 1")
print("=" * 50)
print(f"Average Solve Time (s): {avg_time:.4f}" if solve_times else "Average Solve Time (s): Not available")
print(f"Average Objective: {np.mean(valid_objs):.6f}" if valid_objs else "Average Objective: Not available")
print(f"Average Variables: {avg_vars:.0f}" if num_vars else "Average Variables: Not available")
print(f"Average Constraints: {avg_cons:.0f}" if num_cons else "Average Constraints: Not available")
print(f"Statuses: {set(solve_status)}")
print("=" * 50)

print(f"\nPhase 1 produced {len(start_x_list)} solutions (one per seed)")
for i, seed_sol in enumerate(start_x_list):
    assigned = sum(1 for v in seed_sol.values() if v > 0.5)
    obj_str = f"{objectives[i]:.6f}" if not np.isnan(objectives[i]) else "No solution"
    print(
        f"  Seed {seeds[i]}: Time={solve_times[i]:.2f}s, "
        f"Obj={obj_str}, Assigned room-time triples={assigned}, Status={solve_status[i]}"
    )


SOLVER METRICS SUMMARY - PHASE 1
Average Solve Time (s): 2249.2303
Average Objective: 0.000000
Average Variables: 53573
Average Constraints: 134885
Statuses: {('ok', 'optimal')}

Phase 1 produced 5 solutions (one per seed)
  Seed 1: Time=2463.53s, Obj=0.000000, Assigned room-time triples=200, Status=('ok', 'optimal')
  Seed 2: Time=2055.96s, Obj=0.000000, Assigned room-time triples=200, Status=('ok', 'optimal')
  Seed 5: Time=2361.75s, Obj=0.000000, Assigned room-time triples=200, Status=('ok', 'optimal')
  Seed 7: Time=2240.32s, Obj=0.000000, Assigned room-time triples=200, Status=('ok', 'optimal')
  Seed 10: Time=2124.59s, Obj=0.000000, Assigned room-time triples=200, Status=('ok', 'optimal')


### OTHER

In [ ]:
# ---- Model ----
m1 = pyo.ConcreteModel()

# ---- Sets ----
m1.E = pyo.Set(initialize=E)
m1.T = pyo.Set(initialize=T, ordered=True)
m1.R = pyo.Set(initialize=R)

m1.Teachers = pyo.Set(initialize=all_teachers)
m1.Classes = pyo.Set(initialize=all_classes)

# feasible (event, start) pairs
m1.ES_index = pyo.Set(
    dimen=2,
    initialize=[(e, ts) for e in E for ts in feasible_starts[e]]
)

# feasible (event, start, room) triples
m1.X_index = pyo.Set(
    dimen=3,
    initialize=[
        (e, ts, r)
        for e in E
        for ts in feasible_starts[e]
        for r in feasible_rooms[e]
    ]
)

# ---- Sparse occupancy maps ----
occupies_at_time_room = defaultdict(list)
teacher_occ = defaultdict(list)
class_occ = defaultdict(list)

for (e, ts, r) in m1.X_index:
    occ = occ_times[(e, ts)]

    for t in occ:
        occupies_at_time_room[(t, r)].append((e, ts, r))

        for teacher in as_list(event_to_teachers[e]):
            teacher_occ[(teacher, t)].append((e, ts, r))

        for cls in as_list(event_to_classes[e]):
            class_occ[(cls, t)].append((e, ts, r))

# optional helper: start chosen regardless of room
start_terms = defaultdict(list)
for (e, ts, r) in m1.X_index:
    start_terms[(e, ts)].append((e, ts, r))

# ---- Variables ----
m1.x = pyo.Var(m1.X_index, domain=pyo.Binary)

m1.start_sum = pyo.Expression(
    m1.ES_index,
    rule=lambda m, e, ts: sum(m.x[i] for i in start_terms[(e, ts)])
)

# ---- Hard constraints ----

# each event chooses exactly one feasible start-room triple
def one_start_rule(m, e):
    return sum(
        m.x[e, ts, r]
        for ts in feasible_starts[e]
        for r in feasible_rooms[e]
        if (e, ts, r) in m.X_index
    ) == 1
m1.OneStart = pyo.Constraint(m1.E, rule=one_start_rule)

# room occupancy
def room_conflict_rule(m, r, t):
    terms = [m.x[e, ts, rr] for (e, ts, rr) in occupies_at_time_room.get((t, r), [])]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1
m1.RoomConflict = pyo.Constraint(m1.R, m1.T, rule=room_conflict_rule)

# teacher occupancy
def teacher_conflict_rule(m, teacher, t):
    terms = [m.x[e, ts, r] for (e, ts, r) in teacher_occ.get((teacher, t), [])]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1
m1.TeacherConflict = pyo.Constraint(m1.Teachers, m1.T, rule=teacher_conflict_rule)

# class occupancy
def class_conflict_rule(m, cls, t):
    terms = [m.x[e, ts, r] for (e, ts, r) in class_occ.get((cls, t), [])]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1
m1.ClassConflict = pyo.Constraint(m1.Classes, m1.T, rule=class_conflict_rule)

# hard spread: at most one event of same course per day
event_to_course = dict(zip(df_events["event_id"], df_events["course_reference"]))
days = sorted(df_times["day_ref"].dropna().unique().tolist())
courses = sorted(set(event_to_course[e] for e in E))

course_day_terms = defaultdict(list)
for (e, ts) in m1.ES_index:
    course_day_terms[(event_to_course[e], time_to_day[ts])].append((e, ts))

m1.COURSES = pyo.Set(initialize=courses)
m1.DAYS = pyo.Set(initialize=days)

def event_spread_max_rule(m, c, d):
    terms = course_day_terms.get((c, d), [])
    if not terms:
        return pyo.Constraint.Skip
    return sum(m.start_sum[e, ts] for (e, ts) in terms) <= 1
m1.EventSpreadMax = pyo.Constraint(m1.COURSES, m1.DAYS, rule=event_spread_max_rule)


# 5) Room-group capacities
#roomgroup_cap = {
#    "gr_Room_X": 1,
#    "gr_Room_Y": 2,
#    "gr_Room_Z": 9,
#}

#m1.G = pyo.Set(initialize=roomgroup_cap.keys())


#m1.RoomGroupCapacity = pyo.Constraint(m1.G, m1.T, rule=roomgroup_capacity_rule)


# hard objective
m1.obj = pyo.Objective(expr=0, sense=pyo.minimize)


In [18]:
# Solve
solver = pyo.SolverFactory("gurobi")
# Optional: solver options
# solver.options["TimeLimit"] = 60
# solver.options["MIPGap"] = 0.0

res = solver.solve(m1, tee=True)

print(res.solver.termination_condition)

Read LP format model from file C:\Users\asher\AppData\Local\Temp\tmpvjrfv83a.pyomo.lp
Reading time = 0.16 seconds
x1: 1676 rows, 17365 columns, 121376 nonzeros
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i5-10210U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 1676 rows, 17365 columns and 121376 nonzeros (Min)
Model fingerprint: 0xf9b99088
Model has 0 linear objective coefficients
Variable types: 1 continuous, 17364 integer (17364 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 9e+00]
Presolve removed 698 rows and 1 columns
Presolve time: 0.31s
Presolved: 978 rows, 17364 columns, 105167 nonzeros
Variable types: 0 continuous, 17364 integer (17364 binary)

Root relaxation: objective 0.000000e+00, 3807 iterations

In [16]:
# Solve
solver = pyo.SolverFactory("gurobi")
# Optional: solver options
# solver.options["TimeLimit"] = 60
# solver.options["MIPGap"] = 0.0

res = solver.solve(m1, tee=True)

print(res.solver.termination_condition)

Read LP format model from file C:\Users\asher\AppData\Local\Temp\tmp6afq51l_.pyomo.lp
Reading time = 0.09 seconds
x1: 1616 rows, 17365 columns, 101330 nonzeros
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i5-10210U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 1616 rows, 17365 columns and 101330 nonzeros (Min)
Model fingerprint: 0x8a949a6d
Model has 0 linear objective coefficients
Variable types: 1 continuous, 17364 integer (17364 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 658 rows and 1 columns
Presolve time: 0.18s
Presolved: 958 rows, 17364 columns, 86645 nonzeros
Variable types: 0 continuous, 17364 integer (17364 binary)

Root relaxation: objective 0.000000e+00, 3901 iterations,

In [9]:
# Explicit 2
seeds = [10]

solve_times = []
objectives = []
num_vars = []
num_cons = []
solve_status = []
start_x_list = []

import time as time_module

def clear_all_var_values(model):
    for v in model.component_data_objects(pyo.Var, active=True):
        v.set_value(None, skip_validation=True)

for seed in seeds:
    clear_all_var_values(m1)

    solver = pyo.SolverFactory("gurobi")
    solver.options["TimeLimit"] = 600
    solver.options["Seed"] = seed
    solver.options["Threads"] = 1

    # Manual timing
    start_time = time_module.time()
    res = solver.solve(m1, tee=False)
    elapsed_time = time_module.time() - start_time

    solve_times.append(elapsed_time)
    objectives.append(pyo.value(m1.obj))
    num_vars.append(m1.nvariables())
    num_cons.append(m1.nconstraints())

    solve_status.append(res.solver.termination_condition)

    # Store solution
    seed_solution = {}
    for (e, ts) in m1.X_index:
        val = pyo.value(m1.x[e, ts], exception=False)
        seed_solution[(e, ts)] = 0.0 if val is None else float(val)
    start_x_list.append(seed_solution)

# Keep compatibility with later cells that expect start_x / res
start_x = start_x_list[0]
res = res

# Calculate averages
avg_time = sum(solve_times) / len(solve_times)
avg_obj = sum(objectives) / len(objectives)
avg_vars = sum(num_vars) / len(num_vars)
avg_cons = sum(num_cons) / len(num_cons)

print("=" * 50)
print("SOLVER METRICS SUMMARY - PHASE 1")
print("=" * 50)
print(f"Average Solve Time (s): {avg_time:.4f}")
print(f"Average Objective: {avg_obj:.6f}")
print(f"Average Variables: {avg_vars:.0f}")
print(f"Average Constraints: {avg_cons:.0f}")
print(f"Termination Status: {set(solve_status)}")
print("=" * 50)

print(f"\nPhase 1 produced {len(start_x_list)} solutions (one per seed)")
for i, seed_sol in enumerate(start_x_list):
    scheduled = sum(1 for v in seed_sol.values() if v > 0)
    print(f"  Seed {seeds[i]}: {scheduled} events scheduled in {solve_times[i]:.2f}s, Obj={objectives[i]:.6f}")


ValueError: Cannot load a SolverResults object with bad status: aborted

### 2b: Model 1

In [8]:
# ---- Model ----
m1 = pyo.ConcreteModel()

# ---- Sets ----
m1.E = pyo.Set(initialize=E) # Set of events
m1.T = pyo.Set(initialize=T, ordered=True) # Set of timeslots (ordered)
m1.R = pyo.Set(initialize=R) # Set of rooms

m1.Teachers = pyo.Set(initialize=all_teachers) # set of all teachers
m1.Classes = pyo.Set(initialize=all_classes) # set of all classes

# standard set up
m1.X_index = pyo.Set(
    dimen=2,
    initialize=[(e, ts) for e in E for ts in feasible_starts[e]]
)



# ---- Decision Variables ----
# Start-time decision variables:
# x[e, ts, r] = 1 if event e starts at time ts in room r
# Only define variables for feasible starts (and all rooms) (THIS SPEEDS UP MODEL)
m1.x = pyo.Var(m1.X_index, domain=pyo.Binary)

### BFCs

In [9]:

# Standard set up

# 1) Each event scheduled exactly once (one start time)   (covers constraints 4)
def one_start_rule(m, e):
    return sum(m.x[e, ts] for ts in feasible_starts[e]) == 1
m1.OneStart = pyo.Constraint(m1.E, rule=one_start_rule)

# 2) No room conflicts: at most one event can occupy a room at a time
#def room_conflict_rule(m, r, t):
#    return sum(m.x[e, ts, r] for (e, ts) in occupies_at_time[t]) <= 1
#m.RoomConflict = pyo.Constraint(m.R, m.T, rule=room_conflict_rule)

# 3) No teacher conflicts: a teacher can teach at most one event at a time
def teacher_conflict_rule(m, teacher, t):
    terms = [
        m.x[e, ts]
        for (e, ts) in occupies_at_time.get(t, [])
        if teacher in as_list(event_to_teachers[e])
    ]
    if not terms:
        return pyo.Constraint.Skip   # or pyo.Constraint.Feasible
    return sum(terms) <= 1

m1.TeacherConflict = pyo.Constraint(all_teachers, m1.T, rule=teacher_conflict_rule)

# 4) No class conflicts: a class can attend at most one event at a time
def class_conflict_rule(m, cls, t):
    terms = [
        m.x[e, ts]
        for (e, ts) in occupies_at_time.get(t, [])
        if cls in event_to_classes[e]
    ]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1

m1.ClassConflict = pyo.Constraint(all_classes, m1.T, rule=class_conflict_rule)

# Constraint 14 covered by these

#5 ) Make sure that no timeperiod has more events of a certain roomgroup than we have rooms of that group

# Room-group capacities
#roomgroup_cap = {
#    "gr_Room_X": 1,
#    "gr_Room_Y": 2,
#    "gr_Room_Z": 9,
#}

#m1.G = pyo.Set(initialize=roomgroup_cap.keys())


#m1.RoomGroupCapacity = pyo.Constraint(m1.G, m1.T, rule=roomgroup_capacity_rule)



### Hard  Constraints

In [10]:
### 3) Events that are part of the same course must be on different days (constraint 13)


# 1) Map event -> course_reference
event_to_course = dict(zip(df_events["event_id"], df_events["course_reference"]))

# 2) Build course groups (gr_C001 ... gr_C150)
courses = sorted(set(event_to_course[e] for e in E))

# 3) Days (time groups) from df_times (gr_Mo, gr_Tu, ...)
days = sorted(df_times["day_ref"].dropna().unique().tolist())

# 4) Collect relevant x vars by (course, day)
#    key -> list of all feasible (e, ts) combinations for given course and day
course_day_terms = defaultdict(list)

for (e, ts) in m1.X_index:
    c = event_to_course[e]
    d = time_to_day[ts]   # day of the start time
    course_day_terms[(c, d)].append((e, ts))

# 5) Define Pyomo sets for indexing
m1.COURSES = pyo.Set(initialize=courses)
m1.DAYS = pyo.Set(initialize=days)

# 6) Max-per-day constraint (Maximum = 1)
def event_spread_max_rule(m, c, d):
    terms = course_day_terms.get((c, d), [])
    if not terms:
        return pyo.Constraint.Skip
    return sum(m.x[e, ts] for (e, ts) in terms) <= 1 # at most one event for every course each day

m1.EventSpreadMax = pyo.Constraint(m1.COURSES, m1.DAYS, rule=event_spread_max_rule)



# Example from your data: min=0, max=1 for each day
#day_min = {d: 0 for d in days}
#day_max = {d: 1 for d in days}



#m.EventSpreadMin = pyo.Constraint(m.COURSES, m.DAYS, rule=event_spread_min_rule)
#m.EventSpreadMax = pyo.Constraint(m.COURSES, m.DAYS, rule=event_spread_max_rule)



In [ ]:
# Room constraints

# 1) Each event gets exactly one room
def one_room_rule(m, e):
    return sum(m.x[e, r] for r in feasible_rooms[e]) == 1
m2.OneRoom = pyo.Constraint(m2.E, rule=one_room_rule)


# 2) No room conflicts: at most one event can occupy a room at a time
def room_conflict_rule(m, r, t):
    terms = [m.x[e, r] for e in events_occupying_t.get(t, []) if (e, r) in m.X_index]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1

m2.RoomConflict = pyo.Constraint(m2.R, m2.T, rule=room_conflict_rule)


# Note events will automaticallly be allocated to correct room groups as it is pre-defined by "feasible_rooms" in the preprocessing

### Solve

In [11]:
# ---- Objective ---- (hard constraints only)
m1.obj = pyo.Objective(expr=0, sense=pyo.minimize)

In [12]:
print("vars =", m1.nvariables())
print("cons =", m1.nconstraints())


vars = 2650
cons = 1376


In [13]:
seeds = [10, 20, 30, 40, 50]

solve_times = []
objectives = []
num_vars = []
num_cons = []
solve_status = []
start_x_list = []

import time as time_module

def clear_all_var_values(model):
    for v in model.component_data_objects(pyo.Var, active=True):
        v.set_value(None, skip_validation=True)

for seed in seeds:
    clear_all_var_values(m1)

    solver = pyo.SolverFactory("gurobi")
    solver.options["TimeLimit"] = 600
    solver.options["Seed"] = seed
    solver.options["Threads"] = 1

    # Manual timing
    start_time = time_module.time()
    res = solver.solve(m1, tee=False)
    elapsed_time = time_module.time() - start_time

    solve_times.append(elapsed_time)
    objectives.append(pyo.value(m1.obj))
    num_vars.append(m1.nvariables())
    num_cons.append(m1.nconstraints())

    solve_status.append(res.solver.termination_condition)

    # Store solution
    seed_solution = {}
    for (e, ts) in m1.X_index:
        val = pyo.value(m1.x[e, ts], exception=False)
        seed_solution[(e, ts)] = 0.0 if val is None else float(val)
    start_x_list.append(seed_solution)

# Keep compatibility with later cells that expect start_x / res
start_x = start_x_list[0]
res = res

# Calculate averages
avg_time = sum(solve_times) / len(solve_times)
avg_obj = sum(objectives) / len(objectives)
avg_vars = sum(num_vars) / len(num_vars)
avg_cons = sum(num_cons) / len(num_cons)

print("=" * 50)
print("SOLVER METRICS SUMMARY - PHASE 1")
print("=" * 50)
print(f"Average Solve Time (s): {avg_time:.4f}")
print(f"Average Objective: {avg_obj:.6f}")
print(f"Average Variables: {avg_vars:.0f}")
print(f"Average Constraints: {avg_cons:.0f}")
print(f"Termination Status: {set(solve_status)}")
print("=" * 50)

print(f"\nPhase 1 produced {len(start_x_list)} solutions (one per seed)")
for i, seed_sol in enumerate(start_x_list):
    scheduled = sum(1 for v in seed_sol.values() if v > 0)
    print(f"  Seed {seeds[i]}: {scheduled} events scheduled in {solve_times[i]:.2f}s, Obj={objectives[i]:.6f}")


SOLVER METRICS SUMMARY - PHASE 1
Average Solve Time (s): 15.2464
Average Objective: 0.000000
Average Variables: 2650
Average Constraints: 1376
Termination Status: {<TerminationCondition.optimal: 'optimal'>}

Phase 1 produced 5 solutions (one per seed)
  Seed 10: 169 events scheduled in 8.89s, Obj=0.000000
  Seed 20: 169 events scheduled in 16.12s, Obj=0.000000
  Seed 30: 169 events scheduled in 26.96s, Obj=0.000000
  Seed 40: 169 events scheduled in 11.87s, Obj=0.000000
  Seed 50: 169 events scheduled in 12.39s, Obj=0.000000


In [39]:
# ---- Solve ---

solver = pyo.SolverFactory("gurobi")
#solver.options["TimeLimit"] = 1000
# solver.options["MIPGap"] = 0.0

res = solver.solve(m1, tee=True)

print(res.solver.termination_condition) 

Read LP format model from file C:\Users\asher\AppData\Local\Temp\tmpiuea4zkj.pyomo.lp
Reading time = 0.04 seconds
x1: 2936 rows, 4151 columns, 21894 nonzeros
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i5-10210U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 2936 rows, 4151 columns and 21894 nonzeros (Min)
Model fingerprint: 0x308a67fd
Model has 0 linear objective coefficients
Variable types: 1501 continuous, 2650 integer (2650 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 9e+00]
Presolve removed 2158 rows and 1501 columns
Presolve time: 0.03s
Presolved: 778 rows, 2650 columns, 12704 nonzeros
Variable types: 0 continuous, 2650 integer (2650 binary)

Root relaxation: objective 0.000000e+00, 2626 iterations, 0

In [40]:
from pyomo.opt import TerminationCondition as TC

tc1 = res.solver.termination_condition
print("Phase 1 termination:", tc1)

if tc1 not in [TC.optimal, TC.feasible]:
    raise RuntimeError(f"Phase 1 did not find a feasible solution: {tc1}")


Phase 1 termination: optimal


In [41]:
start_x = {}
for idx in m1.X_index:
    v = pyo.value(m1.x[idx], exception=False)
    start_x[idx] = 0.0 if v is None else float(v)

start_x

{('Event_C001_1', 'Mo_1'): -0.0,
 ('Event_C001_1', 'Mo_2'): -0.0,
 ('Event_C001_1', 'Mo_3'): -0.0,
 ('Event_C001_1', 'Mo_4'): -0.0,
 ('Event_C001_1', 'Tu_1'): -0.0,
 ('Event_C001_1', 'Tu_2'): 1.0,
 ('Event_C001_1', 'Tu_3'): 0.0,
 ('Event_C001_1', 'Tu_4'): -0.0,
 ('Event_C001_1', 'We_1'): -0.0,
 ('Event_C001_1', 'We_2'): -0.0,
 ('Event_C001_1', 'We_3'): -0.0,
 ('Event_C001_1', 'We_4'): -0.0,
 ('Event_C001_1', 'Th_1'): -0.0,
 ('Event_C001_1', 'Th_2'): -0.0,
 ('Event_C001_1', 'Th_3'): -0.0,
 ('Event_C001_1', 'Th_4'): -0.0,
 ('Event_C001_1', 'Fr_1'): -0.0,
 ('Event_C001_1', 'Fr_2'): -0.0,
 ('Event_C001_1', 'Fr_3'): -0.0,
 ('Event_C001_1', 'Fr_4'): -0.0,
 ('Event_C002_1', 'Mo_1'): -0.0,
 ('Event_C002_1', 'Mo_2'): -0.0,
 ('Event_C002_1', 'Mo_3'): 1.0,
 ('Event_C002_1', 'Mo_4'): -0.0,
 ('Event_C002_1', 'Tu_1'): -0.0,
 ('Event_C002_1', 'Tu_2'): -0.0,
 ('Event_C002_1', 'Tu_3'): -0.0,
 ('Event_C002_1', 'Tu_4'): 0.0,
 ('Event_C002_1', 'We_1'): -0.0,
 ('Event_C002_1', 'We_2'): -0.0,
 ('Event_C002_

### 2c: Model 2

### 2d: Model 3

In [37]:
# ---- Model ----
m2 = pyo.ConcreteModel()

# ---- Sets ----
m2.E = pyo.Set(initialize=E) # Set of events
m2.T = pyo.Set(initialize=T, ordered=True) # Set of timeslots (ordered)
m2.R = pyo.Set(initialize=R) # Set of rooms

# standard set up
m2.X_index = pyo.Set(
    dimen=2,
    initialize=[(e, r) for e in E for r in feasible_rooms[e]]
)



events_occupying_t = defaultdict(list)
for e in E:
    ts = chosen_start[e]
    for t in occ_times[(e, ts)]:
        events_occupying_t[t].append(e)

# ---- Decision Variables ----
# Start-time decision variables:
# x[e, ts, r] = 1 if event e starts at time ts in room r
# Only define variables for feasible starts (and all rooms) (THIS SPEEDS UP MODEL)
m2.x = pyo.Var(m2.X_index, domain=pyo.Binary)

In [38]:
# Standard set up

# 1) Each event gets exactly one room
def one_room_rule(m, e):
    return sum(m.x[e, r] for r in feasible_rooms[e]) == 1
m2.OneRoom = pyo.Constraint(m2.E, rule=one_room_rule)


# 2) No room conflicts: at most one event can occupy a room at a time
def room_conflict_rule(m, r, t):
    terms = [m.x[e, r] for e in events_occupying_t.get(t, []) if (e, r) in m.X_index]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1

m2.RoomConflict = pyo.Constraint(m2.R, m2.T, rule=room_conflict_rule)


# Note events will automaticallly be allocated to correct room groups as it is pre-defined by "feasible_rooms" in the preprocessing

In [39]:
# ---- Objective ---- 
m2.obj = pyo.Objective(expr=0, sense=pyo.minimize)

In [40]:
# ---- Solve ---

solver = pyo.SolverFactory("gurobi")
# solver.options["TimeLimit"] = 1000
# solver.options["MIPGap"] = 0.0

res = solver.solve(m2, tee=True)

print(res.solver.termination_condition) 

Read LP format model from file C:\Users\asher\AppData\Local\Temp\tmpb2px3rb2.pyomo.lp
Reading time = 0.04 seconds
x1: 409 rows, 1113 columns, 2472 nonzeros
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i5-10210U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 409 rows, 1113 columns and 2472 nonzeros (Min)
Model fingerprint: 0x484a525e
Model has 0 linear objective coefficients
Variable types: 1 continuous, 1112 integer (1112 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Found heuristic solution: objective 0.0000000

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 1: 0 

Optimal solution found (tolerance 1.00e-04)
Best

### Results

In [41]:
# Helper to safely read var values
def val01(v):
    vv = pyo.value(v)
    return 0 if vv is None else vv

# event -> chosen room from m2
chosen_room = {}
for (e, r) in m2.X_index:
    if val01(m2.x[e, r]) > 0.5:
        chosen_room[e] = r

assignments = []

for e in E:
    ts = chosen_start[e]               # from m1 solution
    r  = chosen_room.get(e, None)      # from m2 solution
    occ = occ_times[(e, ts)]

    assignments.append({
        "event": e,
        "day": time_to_day[ts],
        "start": ts,
        "room": r,
        "duration": int(event_to_duration[e]),
        "times_occupied": occ,
    })

assignments_sorted = sorted(
    assignments,
    key=lambda row: (row["day"], T_index[row["start"]], row["room"] if row["room"] is not None else "", row["event"])
)

for row in assignments_sorted:
    print(f"{row['event']:>12} | {row['day']} | start={row['start']} | room={row['room']} | occ={row['times_occupied']}")


Event_C077_1 | gr_Fr | start=Fr_1 | room=RX_1 | occ=['Fr_1']
Event_C025_1 | gr_Fr | start=Fr_1 | room=RY_1 | occ=['Fr_1']
Event_C126_1 | gr_Fr | start=Fr_1 | room=RY_2 | occ=['Fr_1']
Event_C004_1 | gr_Fr | start=Fr_1 | room=RZ_1 | occ=['Fr_1']
Event_C047_1 | gr_Fr | start=Fr_1 | room=RZ_5 | occ=['Fr_1']
Event_C040_1 | gr_Fr | start=Fr_1 | room=RZ_6 | occ=['Fr_1']
Event_C122_1 | gr_Fr | start=Fr_1 | room=RZ_7 | occ=['Fr_1', 'Fr_2']
Event_C101_1 | gr_Fr | start=Fr_1 | room=RZ_8 | occ=['Fr_1']
Event_C067_1 | gr_Fr | start=Fr_1 | room=RZ_9 | occ=['Fr_1']
Event_C124_1 | gr_Fr | start=Fr_2 | room=RX_1 | occ=['Fr_2']
Event_C118_1 | gr_Fr | start=Fr_2 | room=RY_1 | occ=['Fr_2']
Event_C069_1 | gr_Fr | start=Fr_2 | room=RY_2 | occ=['Fr_2']
Event_C031_1 | gr_Fr | start=Fr_2 | room=RZ_1 | occ=['Fr_2']
Event_C049_2 | gr_Fr | start=Fr_2 | room=RZ_3 | occ=['Fr_2']
Event_C104_1 | gr_Fr | start=Fr_2 | room=RZ_5 | occ=['Fr_2']
Event_C009_1 | gr_Fr | start=Fr_2 | room=RZ_6 | occ=['Fr_2']
Event_C079_1 | g

In [42]:
df_sol = pd.DataFrame(assignments_sorted)
df_sol

,event,day,start,room,duration,times_occupied
0,Event_C077_1,gr_Fr,Fr_1,RX_1,1,[Fr_1]
1,Event_C025_1,gr_Fr,Fr_1,RY_1,1,[Fr_1]
2,Event_C126_1,gr_Fr,Fr_1,RY_2,1,[Fr_1]
3,Event_C004_1,gr_Fr,Fr_1,RZ_1,1,[Fr_1]
4,Event_C047_1,gr_Fr,Fr_1,RZ_5,1,[Fr_1]
...,...,...,...,...,...,...
164,Event_C006_2,gr_We,We_4,RZ_3,1,[We_4]
165,Event_C060_1,gr_We,We_4,RZ_6,1,[We_4]
166,Event_C045_1,gr_We,We_4,RZ_7,1,[We_4]
167,Event_C115_1,gr_We,We_4,RZ_8,1,[We_4]


In [43]:
print("Total events:", len(E))
print("Assigned events:", len(set(r["event"] for r in assignments_sorted)))
print("Total scheduled triplets (should equal #events if exactly-once):", len(assignments_sorted))

Total events: 169
Assigned events: 169
Total scheduled triplets (should equal #events if exactly-once): 169


In [44]:
def _as_list(x):
    if x is None:
        return []
    if isinstance(x, (list, tuple, set)):
        return list(x)
    return [x]


# Step

def extract_selected_m1(m1, tol=0.5):
    sel = []
    for (e, ts) in m1.X_index:
        v = pyo.value(m1.x[e, ts], exception=False)
        if v is not None and v > tol:
            sel.append((e, ts))
    return sel

def chosen_start_from_m1(m1, feasible_starts, tol=0.5):
    chosen = {}
    for e in m1.E:
        picks = [ts for ts in feasible_starts[e] if pyo.value(m1.x[e, ts], exception=False) is not None and pyo.value(m1.x[e, ts]) > tol]
        if len(picks) != 1:
            raise ValueError(f"Event {e}: expected exactly one chosen start, got {picks}")
        chosen[e] = picks[0]
    return chosen

def check_m1_one_start(m1, E):
    sel = extract_selected_m1(m1)
    by_e = defaultdict(int)
    for e, ts in sel:
        by_e[e] += 1
    missing = [e for e in E if by_e[e] == 0]
    multi = [e for e, c in by_e.items() if c > 1]
    return {"ok": len(missing) == 0 and len(multi) == 0, "missing": missing, "multiple": multi}

def check_m1_teacher_conflicts(m1, T, occupies_at_time, event_to_teachers):
    sel = set(extract_selected_m1(m1))
    bad = []
    for t in T:
        load = defaultdict(list)  # teacher -> [(e,ts)]
        for (e, ts) in occupies_at_time.get(t, []):
            if (e, ts) in sel:
                for teacher in _as_list(event_to_teachers[e]):
                    load[teacher].append((e, ts))
        for teacher, assigns in load.items():
            if len(assigns) > 1:
                bad.append((teacher, t, assigns))
    return {"ok": len(bad) == 0, "violations": bad}

def check_m1_class_conflicts(m1, T, occupies_at_time, event_to_classes):
    sel = set(extract_selected_m1(m1))
    bad = []
    for t in T:
        load = defaultdict(list)  # class -> [(e,ts)]
        for (e, ts) in occupies_at_time.get(t, []):
            if (e, ts) in sel:
                for cls in _as_list(event_to_classes[e]):
                    load[cls].append((e, ts))
        for cls, assigns in load.items():
            if len(assigns) > 1:
                bad.append((cls, t, assigns))
    return {"ok": len(bad) == 0, "violations": bad}

def check_m1_roomgroup_capacity(m1, T, occupies_at_time, event_to_roomgroup, roomgroup_cap):
    sel = set(extract_selected_m1(m1))
    bad = []
    for t in T:
        grp_load = defaultdict(list)  # group -> [(e,ts)]
        for (e, ts) in occupies_at_time.get(t, []):
            if (e, ts) in sel:
                g = event_to_roomgroup[e]
                grp_load[g].append((e, ts))
        for g, assigns in grp_load.items():
            cap = roomgroup_cap[g]
            if len(assigns) > cap:
                bad.append((g, t, len(assigns), cap, assigns))
    return {"ok": len(bad) == 0, "violations": bad}

def check_m1_event_spread_by_day(m1, event_to_course, time_to_day, day_max=1):
    sel = extract_selected_m1(m1)
    load = defaultdict(list)  # (course, day) -> [(e,ts)]
    for e, ts in sel:
        c = event_to_course[e]
        d = time_to_day[ts]
        load[(c, d)].append((e, ts))
    bad = [(c, d, assigns) for (c, d), assigns in load.items() if len(assigns) > day_max]
    return {"ok": len(bad) == 0, "violations": bad}

def run_m1_sanity_checks(
    m1, E, T, occupies_at_time, event_to_teachers, event_to_classes,
    event_to_course, time_to_day, event_to_roomgroup, roomgroup_cap
):
    out = {
        "OneStart": check_m1_one_start(m1, E),
        "TeacherConflict": check_m1_teacher_conflicts(m1, T, occupies_at_time, event_to_teachers),
        "ClassConflict": check_m1_class_conflicts(m1, T, occupies_at_time, event_to_classes),
        "RoomGroupCapacity": check_m1_roomgroup_capacity(m1, T, occupies_at_time, event_to_roomgroup, roomgroup_cap),
        "EventSpreadByDay": check_m1_event_spread_by_day(m1, event_to_course, time_to_day, day_max=1),
    }
    summary = pd.DataFrame([
        {
            "check": k,
            "ok": v["ok"],
            "n_violations": len(v.get("violations", [])) + len(v.get("missing", [])) + len(v.get("multiple", []))
        }
        for k, v in out.items()
    ])
    return out, summary


# Step

def extract_selected_m2(m2, tol=0.5):
    sel = []
    for (e, r) in m2.X_index:
        v = pyo.value(m2.x[e, r], exception=False)
        if v is not None and v > tol:
            sel.append((e, r))
    return sel

def check_m2_one_room(m2, E):
    sel = extract_selected_m2(m2)
    by_e = defaultdict(int)
    for e, r in sel:
        by_e[e] += 1
    missing = [e for e in E if by_e[e] == 0]
    multi = [e for e, c in by_e.items() if c > 1]
    return {"ok": len(missing) == 0 and len(multi) == 0, "missing": missing, "multiple": multi}

def check_m2_room_conflicts(m2, T, chosen_start, occ_times):
    sel = set(extract_selected_m2(m2))
    bad = []
    for t in T:
        room_load = defaultdict(list)  # room -> [event]
        for (e, r) in sel:
            ts = chosen_start[e]
            if t in occ_times[(e, ts)]:
                room_load[r].append(e)
        for r, evs in room_load.items():
            if len(evs) > 1:
                bad.append((r, t, evs))
    return {"ok": len(bad) == 0, "violations": bad}

def run_m2_sanity_checks(m2, E, T, chosen_start, occ_times):
    out = {
        "OneRoom": check_m2_one_room(m2, E),
        "RoomConflict": check_m2_room_conflicts(m2, T, chosen_start, occ_times),
    }
    summary = pd.DataFrame([
        {
            "check": k,
            "ok": v["ok"],
            "n_violations": len(v.get("violations", [])) + len(v.get("missing", [])) + len(v.get("multiple", []))
        }
        for k, v in out.items()
    ])
    return out, summary


In [45]:
roomgroup_cap = {"gr_Room_X": 1, "gr_Room_Y": 2, "gr_Room_Z": 9}

m1_results, m1_summary = run_m1_sanity_checks(
    m1, E, T, occupies_at_time, event_to_teachers, event_to_classes,
    event_to_course, time_to_day, event_to_roomgroup, roomgroup_cap
)
display(m1_summary)

chosen_start = chosen_start_from_m1(m1, feasible_starts)

m2_results, m2_summary = run_m2_sanity_checks(
    m2, E, T, chosen_start, occ_times
)
display(m2_summary)


,check,ok,n_violations
0,OneStart,True,0
1,TeacherConflict,True,0
2,ClassConflict,True,0
3,RoomGroupCapacity,True,0
4,EventSpreadByDay,True,0


,check,ok,n_violations
0,OneRoom,True,0
1,RoomConflict,True,0


In [46]:
def soft_violations_report(m1, tol=1e-6, top_n=20):
    rows = []
    detail = {}

    def add_from_var(var_name, label, key_names):
        if not hasattr(m1, var_name):
            return
        v = getattr(m1, var_name)
        recs = []
        for idx in v:
            val = pyo.value(v[idx], exception=False)
            if val is not None and val > tol:
                if not isinstance(idx, tuple):
                    idx = (idx,)
                rec = {k: idx[i] for i, k in enumerate(key_names)}
                rec["deviation"] = float(val)
                rec["squared_cost"] = float(val * val)
                recs.append(rec)
        if recs:
            df = pd.DataFrame(recs).sort_values("deviation", ascending=False)
            detail[label] = df
            rows.append({
                "constraint": label,
                "n_violations": len(df),
                "sum_deviation": df["deviation"].sum(),
                "sum_squares": df["squared_cost"].sum(),
            })

    # LimitBusyTimes-style (or MinMaxADay-style) deviations
    add_from_var("c1_under", "Class1_Under", ["class", "day"])
    add_from_var("c1_over",  "Class1_Over",  ["class", "day"])
    add_from_var("c2_under", "Class2_Under", ["class", "day"])
    add_from_var("c2_over",  "Class2_Over",  ["class", "day"])
    add_from_var("t1_under", "Teacher1_Under", ["teacher", "day"])
    add_from_var("t1_over",  "Teacher1_Over",  ["teacher", "day"])
    add_from_var("t2_under", "Teacher2_Under", ["teacher", "day"])
    add_from_var("t2_over",  "Teacher2_Over",  ["teacher", "day"])

    # JumpPeriods deviations
    add_from_var("idle_class",   "JumpPeriods_Classes",  ["class", "day"])
    add_from_var("idle_teacher", "JumpPeriods_Teachers", ["teacher", "day"])

    summary = pd.DataFrame(rows).sort_values("sum_squares", ascending=False) if rows else pd.DataFrame(
        columns=["constraint", "n_violations", "sum_deviation", "sum_squares"]
    )

    # optional: print top violations per constraint
    for name, df in detail.items():
        print(f"\n{name} (top {top_n})")
        display(df.head(top_n))

    return summary, detail

summary, detail = soft_violations_report(m1, tol=1e-6, top_n=20)
display(summary)



Teacher2_Under (top 20)


,teacher,day,deviation,squared_cost
0,T07,gr_Th,1.0,1.0



JumpPeriods_Classes (top 20)


,class,day,deviation,squared_cost
0,SCB,gr_Mo,1.0,1.0



JumpPeriods_Teachers (top 20)


,teacher,day,deviation,squared_cost
0,T08,gr_Tu,1.0,1.0
1,T09,gr_Th,1.0,1.0
2,T11,gr_Fr,1.0,1.0


,constraint,n_violations,sum_deviation,sum_squares
2,JumpPeriods_Teachers,3,3.0,3.0
0,Teacher2_Under,1,1.0,1.0
1,JumpPeriods_Classes,1,1.0,1.0
